# Ricerca di Short Tandem Repeats

Questo notebook migliora la qualità e individua regioni a bassa complessità di
un dataset FASTQ, applicando in sequenza:
1. Un filtro “Sliding Window Quality” per effettuare il trimming del read al 3’, che consiste
in una finestra scorrevole lunga qualche base che tronca opportunamente il read al 3’ non appena la qualità media all’interno della finestra scende al
di sotto di una certa soglia
2. Un filtro sulla lunghezza, per rimuovere i reads che sono troppo corti

All’interno dei reads rimasti alla fine del precedente filtraggio vengono riconosciuti gli
**Short Tandem Repeats** (STRs) di lunghezza 1, 2 e 3. Uno Short Tandem Repeat è una sottostringa massimale in cui un certo motivo (lungo qualche base)
si ripete un certo numero di volte.

Esempio 1: GGGGGGGGGGGGGGGGGG (il motivo G di una base è ripetuto 18 volte)

Esempio 2: TATATATATATATATA (il motivo TA di due basi è ripetuto 8 volte)

Esempio 3: CAGCAGCAGCAGCAGCAGCAG (il motivo CAG di tre basi è ripetuto 7 volte)

Per ogni read rimasto dopo il filtraggio vengono prodotti in standard output gli eventuali STR, specificando
per ognuno la posizione di inizio nel read, il motivo e quante volte si ripete.

## 0) Importazione dei moduli necessari

In [1]:
from Bio import SeqIO, Seq

In [2]:
import numpy as np

In [3]:
import re

## 1) Parametri di input

- Dimensione della sliding window, espressa in basi, che viene utilizzata per il trimming della sequenza basato sulla qualità
- Soglia che esprime la qualità minima della sottosequenza analizzata dalla sliding window
- Dimensione minima dei read dopo che sono stati filtrati dalla sliding window
- Lunghezza dei motivi che formano uno *Short Tandem Repeats* 
- Lunghezza minima che uno *Short Tandem Repeat* deve avere per essere considerato tale; questa dimensione si riferisce all'intera estensione del STR data dalla ripetizione di un motivo

In [4]:
sliding_window_dim = 5

In [5]:
quality_threshold = 10

In [6]:
read_min_dim = 5

In [7]:
fastq_file_path = "./"

In [8]:
str_pattern_lengths = [1, 2, 3]

In [15]:
min_str_length = 15

## 2) Lettura del dataset
Il primo passaggio da svogere è caricare i read contenuti nel file FASTQ all'interno di un generatore. 

In [23]:
records_generator = SeqIO.parse("./dataset-quality.fastq", "fastq")

## 3) Definizione del filtro Sliding Window
Il filtro *sliding window* deve scorrere una sequenza ed effettuare un trimming nel punto in cui la qualità della sequenza scende al di sotto della soglia configurata. Qeusto algoritmo viene implementato in una funzione che, dati in input:
- La sequenza del read da analizzare
- La dimensione della finestra scorrevole
- La soglia di qualità minima di ogni finestra

Restituisce il read troncato all'utima finestra con qualità maggiore alla soglia. 

In [11]:
def filter_by_quality(sequence, sliding_window_dim, quality_threshold):
    read_quality = sequence.letter_annotations["phred_quality"]
    for i in range(len(sequence) - sliding_window_dim):
        j = i + sliding_window_dim

        sequence_window = sequence.seq[i:j]
        qualities = read_quality[i:j]
        mean_quality = float(np.mean(qualities))

        if mean_quality < quality_threshold:
            return sequence[:i]

    return sequence

## 4) Definizione del filtro sulla lunghezza del read
Questo filtro, implementato da una funzione dedicata, verifica che la lunghezza del read sia maggiore della soglia fornita.

In [12]:
def filter_by_length(sequence, read_min_dim):
    return len(sequence.seq) >= read_min_dim

## 5) Riconoscimento degli Short Tandem Repeats

Data in input la sequenza da analizzare, per ogni posizione nella sequenza viene creata una finestra scorrevole per ogni dimensione ammissibile dei motivi (ad esempio 1, 2 e 3). Fino a quando il valore nella finestra rimane uguale a quello del motivo considerato nella posizione iniziale della finestra, questa viene fatta scorrere verso destra. 

Per ogni Short Tandem Repeat riconosciuto, viene memorizzato un dizionario contenente le seguenti informazioni:
- `pattern`: il motivo di base dell'STR
- `start`: posizione di inizio **zero-based**
- `end`: posizione di fine **zero-based**

La funzione restituisce una lista contenente i dizionari che rappresentano gli Short Tandem Repeats trovati nella sequenza in input. 

In [13]:
def get_short_tandem_repeats(sequence, str_pattern_lengths, min_str_length):
    strs = []
    start_position = 0
    while start_position < len(sequence):
        position_strs = []
        for window_dim in str_pattern_lengths:
            end_position = start_position + window_dim - 1
            
            pattern = sequence[start_position:end_position+1]
            
            j = end_position + 1
            while sequence[j:j + window_dim] == pattern:
                j += window_dim

            if j != end_position + 1:
                str_pattern = sequence[start_position:end_position+1]
                str_start_position = start_position
                str_end_position = j - 1
                str_dim = str_end_position - str_start_position
                
                str_dict = {
                    "pattern": str_pattern,
                    "start": str_start_position,
                    "end": str_end_position
                }
                
                str_dim = str_end_position - str_start_position + 1
                if str_dim < min_str_length:
                    continue

                # Se lo STR della finestra corrente è un suffisso di 
                # un altro STR lo devo ignorare
                if [s for s in strs if s["end"] == str_end_position]:
                    continue

                # Se lo STR è contenuto in un altro str va ignorato
                if [s for s in strs if s["start"] < str_end_position < s["end"]]:
                    continue

                position_strs.append(str_dict)
        
        if position_strs:
            # Per ogni posizione, viene memorizzato lo STR più lungo tra quelli
            # che partono nella posizione considerata
            max_str = max(position_strs, key=lambda x: x["end"]-x["start"])
            strs.append(max_str)

        start_position += 1
    return strs

## 6) Analisi dei reads contenuti nel file

Per ogni read contenuto nel file in input, vengono applicati i filtri sulla qualità e sulla lunghezza definiti precedentemente per ottenere un sottoinsieme di sequenze che soddisfano le condizioni richieste. 

Sui read che hanno superato la verifica dei filtri viene applicato l'algoritmo per trovare gli *short tandem repeats*. 

In [24]:
for record in records_generator:
    filtered_sequence = filter_by_quality(record, sliding_window_dim, quality_threshold)
    if filter_by_length(filtered_sequence, read_min_dim):
        records_strs = get_short_tandem_repeats(filtered_sequence.seq, str_pattern_lengths, min_str_length)
        print("Sequenza:", filtered_sequence.seq)
        if records_strs:
            for str_dict in records_strs:
                repeats_number = int((str_dict["end"] + 1 - str_dict["start"]) / len(str_dict["pattern"]))
                print(f"- {str_dict["pattern"]} ({str_dict["start"]}:{str_dict["end"]}) si ripete per {repeats_number} volte")
        else:
            print("- Non sono stati trovati Short Tandem Repeats nella sequenza")
        print("")

Sequenza: TTCACTTGTGGCAAGGAATATATGTTGTGATGAAATGCACTGACTTGGGACATCTCCCACCCTCGGTGCCACGAATGTTAAGAGCATCTGTGTTACGGGCTAGGTTCACCATTTTCGGCGTTCACCGTGGCCG
- Non sono stati trovati Short Tandem Repeats nella sequenza
-------------------------------
Sequenza: GTCGCCCACCGTCGCAAAACCCATTCACCCGCACTTATAGGCACAGAACTAGACCCGCTGACCGGGGAGAGAGCCTCGGCGCTTCAGTATTACGCCTTCGA
- Non sono stati trovati Short Tandem Repeats nella sequenza
-------------------------------
Sequenza: AGTTCTTAACGTCATGTCCTATATCTGTTTAAACTCTTGCTTCAATGTCCACACTTGTCAACACGCCCCTCCCCAACCGACTTTGTTTCTTGCCTGTCTTAAAGACACTCAGTCAGAATAAGACGTTCGACGTCGGTAGTCCTAGCTATG
- Non sono stati trovati Short Tandem Repeats nella sequenza
-------------------------------
Sequenza: AGTGTCGCCCCTATACCTGCTGCCTTCCATTCCCGATTAGAGGTCGCTTAGCCTTTGTATAATTTCATCAGACCTCGCAAGAAAGGTGGTCCTTCAGGTAATGTTCC
- Non sono stati trovati Short Tandem Repeats nella sequenza
-------------------------------
Sequenza: GTTCGGTGGGGTGCTAGTA
- Non sono stati trovati Short Tandem Repeats nella sequenza
--